## 1. Confirm that a GPU is available



In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU! Runtime -> Change runtime type -> GPU")
print(result.stdout[:400])

## 2. Mount Google Drive



In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_DIR = "/content/drive/MyDrive/weapon_detection"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive: {DRIVE_DIR}")

## 3. Create the project folders



In [ ]:
import os
PROJECT_DIR = "/content/weapon-detection"
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/splits", exist_ok=True)
os.makedirs("results/eda", exist_ok=True)
os.makedirs("results/errors", exist_ok=True)
os.makedirs("results/feature_maps", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)
print("Dirs created.")

## 4. Install the required packages



In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q kagglehub imagehash opencv-python-headless PyYAML scipy tqdm matplotlib seaborn pandas albumentations
print("Dependencies installed.")

## 5. Download YOLOv5



In [ ]:
!git clone --depth 1 --branch v7.0 https://github.com/ultralytics/yolov5.git
!pip install -q -r yolov5/requirements.txt
print("YOLOv5 v7.0 cloned.")

## 6. Download the weapon dataset



In [ ]:
# Upload kaggle.json from https://www.kaggle.com/account -> API
from google.colab import files
print("Upload your kaggle.json")
uploaded = files.upload()
import os, shutil
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
import kagglehub
path = kagglehub.dataset_download("raghavnanjappan/weapon-dataset-for-yolov5")
print(f"Downloaded to: {path}")
dst = f"{PROJECT_DIR}/data/raw/dataset"
if not os.path.exists(dst):
    shutil.copytree(path, dst)
print(f"Dataset at: {dst}")

## 7. Check for the project configuration


In [ ]:
# Paste config.yaml content
cfg = open("config.yaml").read() if os.path.exists("config.yaml") else None
if cfg:
    print("config.yaml already present")
else:
    print("Upload config.yaml from your local project")

## 8. Audit the dataset split


In [ ]:
!python scripts/audit_split.py
print("Phase 1 done.")

## 9. Run a short smoke test



In [ ]:
!python scripts/train.py --smoke
print("Smoke test passed.")

## 10. Train YOLOv5s



In [ ]:
# Full training — ~2-4h on T4
!python yolov5/train.py --data data.yaml --weights yolov5s.pt --img 640 --epochs 100 --batch 16 --workers 2 --project runs/train --name yolov5s_baseline --patience 20 --seed 42 --device 0 --exist-ok
print("Training done.")

## 11. Evaluate the trained model



In [ ]:
!python scripts/evaluate.py
print("Evaluation done.")

## 12. Run additional experiments



In [ ]:
!python scripts/experiments.py --epochs 30
print("Experiments done.")

## 13. Inspect the trained model



In [ ]:
!python scripts/inspect_model.py
print("Inspection done.")

## 14. Save results and checkpoints to Google Drive


In [ ]:
import shutil, glob
shutil.copytree(f"{PROJECT_DIR}/results", f"{DRIVE_DIR}/results", dirs_exist_ok=True)
for w in glob.glob(f"{PROJECT_DIR}/runs/train/yolov5s_baseline/weights/*.pt"):
    shutil.copy(w, DRIVE_DIR)
shutil.make_archive("/content/results", "zip", f"{PROJECT_DIR}/results")
from google.colab import files
files.download("/content/results.zip")
print("Done.")